<a href="https://colab.research.google.com/github/Timbo3399/INFM_SOTA2/blob/main/notebooks/workbook_marvin_training_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 – Training mit Wikitext (einfach)

**Autor:** Marvin Ebi  
**Ziel:** Einfaches Training wie auf den Folien: Wikitext aus Hugging Face laden, mit `BPEDataset` Token-Fenster erzeugen, per PyTorch `DataLoader` trainieren und den Loss auswerten.

In Colab installiert `uv` die Notebook-Pakete in den aktiven Kernel. Eine separate venv wäre nicht automatisch der Notebook-Kernel und könnte Colabs GPU-PyTorch umgehen. Checkpoints werden in Colab nach Google Drive geschrieben (beim ersten Lauf musst du den Zugriff erlauben); außerhalb von Colab landen sie im lokalen Ordner `checkpoints/`.


In [ ]:
from pathlib import Path
import os
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = Path('/content/INFM_SOTA2')
    if not REPO_DIR.exists():
        import subprocess
        subprocess.run(
            ['git', 'clone', '--depth', '1', 'https://github.com/Timbo3399/INFM_SOTA2.git', str(REPO_DIR)],
            check=True,
        )
    os.chdir(REPO_DIR)

    # Keep Colab's GPU-enabled torch; install notebook dependencies in its active kernel.
    !python -m pip install -q uv
    !uv pip install --system --no-deps -e .
    !uv pip install --system "datasets==5.1.0" "huggingface-hub==2.4.0" "fsspec==2026.7.0" "tiktoken==0.14.0" "wandb==0.28.1" "nbformat==5.11.1"
else:
    REPO_DIR = next(
        (p for p in (Path.cwd(), *Path.cwd().parents)
         if (p / 'pyproject.toml').is_file() and (p / 'src' / 'infm_sota2').is_dir()),
        None,
    )
    if REPO_DIR is None:
        raise FileNotFoundError('Projektwurzel nicht gefunden. Notebook im Repo starten.')
    os.chdir(REPO_DIR)
    sys.path.insert(0, str(REPO_DIR / 'src'))

print(f'Projektpfad: {REPO_DIR}')


## 1. Konfiguration und Datensatz laden

Datensatz, Splits und Trainingsparameter werden zentral in `CONFIG` gepflegt. Bei langen Läufen speichert das Notebook standardmäßig alle 100 Optimizer-Schritte einen wiederaufnehmbaren Checkpoint; die Häufigkeit lässt sich dort anpassen.


In [ ]:
from datasets import load_dataset

CONFIG = {
    'dataset': {
        'path': 'Salesforce/wikitext',
        'name': 'wikitext-103-raw-v1',
        'train_split': 'train[:]',
        'validation_split': 'validation[:]',
    },
    'sequence_length': 64,
    'batch_size': 4,
    'epochs': 1,
    'learning_rate': 0.001,
    'weight_decay': 0.1,
    'checkpoint_every_steps': 100000,
    'log_every_steps': 100,
    'seed': 42,
    'resume_checkpoint': None,  # None = neu starten, 'latest' = neuesten Lauf fortsetzen, oder konkreter Checkpoint-Pfad
    'model': {
        'emb_dim': 64,
        'n_layers': 2,
        'n_heads': 4,
        'drop_rate': 0.1,
        'qkv_bias': True,
    },
}

DATASET_CONFIG = CONFIG['dataset']
train_hf = load_dataset(
    DATASET_CONFIG['path'],
    DATASET_CONFIG['name'],
    split=DATASET_CONFIG['train_split'],
)
val_hf = load_dataset(
    DATASET_CONFIG['path'],
    DATASET_CONFIG['name'],
    split=DATASET_CONFIG['validation_split'],
)

print(train_hf)
print(val_hf)


## 2. BPEDataset und DataLoader bauen

Die Hugging-Face-Zeilen werden zu einem Text zusammengefügt. `BPEDataset` tokenisiert ihn und erzeugt Eingabe-/Zielsequenzen mit einem Token Versatz.


In [ ]:
import torch
from torch.utils.data import DataLoader
from infm_sota2.data import BPEDataset

SEQ_LEN = CONFIG['sequence_length']
BATCH_SIZE = CONFIG['batch_size']

train_text = "\n".join(text for text in train_hf['text'] if text.strip())
val_text = "\n".join(text for text in val_hf['text'] if text.strip())

train_dataset = BPEDataset(train_text, seq_len=SEQ_LEN, stride=SEQ_LEN)
val_dataset = BPEDataset(val_text, seq_len=SEQ_LEN, stride=SEQ_LEN)

tokenizer = train_dataset.tokenizer
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
if not len(train_loader) or not len(val_loader):
    raise ValueError('Trainings- und Validierungsdaten müssen mindestens einen Batch enthalten.')

print(f'Train windows: {len(train_dataset)}')
print(f'Validation windows: {len(val_dataset)}')

x, y = next(iter(train_loader))
print('Batch shape x:', x.shape)
print('Batch shape y:', y.shape)


## 3. Modell trainieren, mit W&B verfolgen und speichern

Beim ersten Start meldest du dich bei W&B an. In Colab kannst du alternativ `WANDB_API_KEY` unter **Secrets** hinterlegen; der Key gehört nicht ins Notebook. In Colab werden Checkpoints in Google Drive gespeichert. `latest` wird regelmäßig während der Epoche und nach jeder Epoche aktualisiert, `best` nach verbessertem Validierungs-Loss. Für eine komfortable Fortsetzung setze `CONFIG['resume_checkpoint'] = 'latest'`; das Notebook wählt den neuesten `-latest.pt`-Checkpoint und setzt auch den W&B-Run fort. Alternativ gib einen konkreten `-latest.pt`-Pfad an. `None` startet bewusst einen neuen Lauf. `CONFIG['epochs']` ist die angestrebte Gesamtzahl an Epochen.


In [ ]:
from datetime import datetime
import os
import torch
import wandb
from torch.utils.data import DataLoader, Subset
from infm_sota2 import GPTModel

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = CONFIG['batch_size']
EPOCHS = CONFIG['epochs']
LEARNING_RATE = CONFIG['learning_rate']
MODEL_CONFIG = {
    **CONFIG['model'],
    'vocab_size': train_dataset.vocab_size,
    'context_length': CONFIG['sequence_length'],
}
DATASET_SLUG = f"{DATASET_CONFIG['path'].split('/')[-1]}-{DATASET_CONFIG['name']}"
run_name = (
    f"{DATASET_SLUG}-gpt-{MODEL_CONFIG['n_layers']}L-{MODEL_CONFIG['n_heads']}H-"
    f"ep{EPOCHS}-bs{BATCH_SIZE}-lr{LEARNING_RATE:g}-"
    f"{datetime.now():%Y%m%d-%H%M%S}"
)

if CONFIG['checkpoint_every_steps'] < 1 or CONFIG['log_every_steps'] < 1:
    raise ValueError('Checkpoint- und Log-Intervalle müssen mindestens 1 sein.')

# Google Drive vor Checkpoint-Zugriffen einbinden, damit sie Colab-Neustarts überstehen.
if IN_COLAB:
    from google.colab import drive, userdata # type: ignore
    drive.mount('/content/drive')
    checkpoint_dir = Path('/content/drive/MyDrive/INFM_SOTA2/checkpoints')
else:
    checkpoint_dir = Path(REPO_DIR) / 'checkpoints' if REPO_DIR is not None else Path.cwd() / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

RESUME_CHECKPOINT = CONFIG['resume_checkpoint']
resume_state = None
if RESUME_CHECKPOINT is not None:
    if RESUME_CHECKPOINT == 'latest':
        available_checkpoints = sorted(
            checkpoint_dir.glob('*-latest.pt'),
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
        if not available_checkpoints:
            raise FileNotFoundError(
                f'Kein -latest.pt-Checkpoint in {checkpoint_dir} gefunden. '
                "Für einen neuen Lauf CONFIG['resume_checkpoint'] = None setzen."
            )
        print('Vorhandene Checkpoints (neuester zuerst):')
        for index, path in enumerate(available_checkpoints, start=1):
            modified_at = datetime.fromtimestamp(path.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')
            print(f'  {index}. {path.name} — geändert {modified_at}')
        RESUME_CHECKPOINT = available_checkpoints[0]
        print(f'Automatisch ausgewählt: {RESUME_CHECKPOINT.name}')
    else:
        RESUME_CHECKPOINT = Path(RESUME_CHECKPOINT)
    if not RESUME_CHECKPOINT.is_file():
        raise FileNotFoundError(f'Checkpoint nicht gefunden: {RESUME_CHECKPOINT}')
    if not RESUME_CHECKPOINT.name.endswith('-latest.pt'):
        raise ValueError('Zum Fortsetzen bitte den Pfad zu einer -latest.pt-Datei angeben.')
    resume_state = torch.load(RESUME_CHECKPOINT, map_location='cpu')
    print(
        f"Checkpoint-Stand: Epoche {resume_state['epoch'] + (1 if resume_state.get('next_step', 0) else 0)}, "
        f"Step {resume_state.get('next_step', 0)}; "
        f"bisher bester Validierungs-Loss: {resume_state.get('best_val_loss', resume_state.get('validation_loss', float('nan'))):.4f}"
    )
    if 'dataset_config' in resume_state and resume_state['dataset_config'] != DATASET_CONFIG:
        raise ValueError('Der Checkpoint gehört zu einer anderen Dataset-Konfiguration.')
    if 'model_config' in resume_state and resume_state['model_config'] != MODEL_CONFIG:
        raise ValueError('Die Modellkonfiguration stimmt nicht mit dem Checkpoint überein.')
    if resume_state.get('next_step', 0) and resume_state.get('batch_size', BATCH_SIZE) != BATCH_SIZE:
        raise ValueError('Batch-Größe beim Fortsetzen einer Epoche nicht ändern.')

if IN_COLAB:
    api_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=api_key) if api_key else wandb.login()
else:
    wandb.login()

wandb_init = {
    'project': 'SOTA_2',
    'name': run_name,
    'config': {
        'dataset': DATASET_CONFIG,
        'sequence_length': CONFIG['sequence_length'],
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'optimizer': 'AdamW',
        'learning_rate': LEARNING_RATE,
        'weight_decay': CONFIG['weight_decay'],
        'checkpoint_every_steps': CONFIG['checkpoint_every_steps'],
        'train_examples': len(train_dataset),
        'validation_examples': len(val_dataset),
        **MODEL_CONFIG,
    },
}
if resume_state and resume_state.get('wandb_run_id'):
    wandb_init.update(id=resume_state['wandb_run_id'], resume='allow')
run = wandb.init(**wandb_init)

torch.manual_seed(CONFIG['seed'])
model = GPTModel(MODEL_CONFIG).to(DEVICE)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=CONFIG['weight_decay'],
)
loss_fn = torch.nn.CrossEntropyLoss()
shuffle_generator = torch.Generator().manual_seed(CONFIG['seed'])

start_epoch = 0
start_step = 0
epoch_indices = None
running_loss = 0.0
running_count = 0
best_val_loss = float('inf')
global_step = 0
if resume_state:
    model.load_state_dict(resume_state['model_state_dict'])
    optimizer.load_state_dict(resume_state['optimizer_state_dict'])
    if resume_state.get('checkpoint_version', 0) >= 2:
        start_epoch = resume_state['epoch']
        start_step = resume_state.get('next_step', 0)
        epoch_indices = resume_state.get('epoch_indices')
        if start_step and epoch_indices is None:
            raise ValueError('Checkpoint enthält keinen Batch-Fortschritt für die laufende Epoche.')
        running_loss = resume_state.get('running_loss', 0.0)
        running_count = resume_state.get('running_count', 0)
        global_step = resume_state.get('global_step', 0)
        best_val_loss = resume_state.get('best_val_loss', float('inf'))
        if 'shuffle_generator_state' in resume_state:
            shuffle_generator.set_state(resume_state['shuffle_generator_state'])
        if 'torch_rng_state' in resume_state:
            torch.set_rng_state(resume_state['torch_rng_state'])
        if DEVICE.type == 'cuda' and resume_state.get('cuda_rng_state'):
            torch.cuda.set_rng_state_all(resume_state['cuda_rng_state'])
    else:
        # Ältere Checkpoints wurden jeweils am Ende einer Epoche gespeichert.
        start_epoch = resume_state['epoch']
        best_val_loss = resume_state.get('validation_loss', float('inf'))
    print(f'Fortsetzung ab Epoche {start_epoch + 1}, Step {start_step + 1}.')

if RESUME_CHECKPOINT is not None:
    latest_checkpoint_path = RESUME_CHECKPOINT
    best_checkpoint_path = RESUME_CHECKPOINT.with_name(
        RESUME_CHECKPOINT.name.replace('-latest.pt', '-best.pt')
    )
else:
    latest_checkpoint_path = checkpoint_dir / f'{DATASET_SLUG}-{run.id}-latest.pt'
    best_checkpoint_path = checkpoint_dir / f'{DATASET_SLUG}-{run.id}-best.pt'

total_steps = len(train_loader)
print(f'Steps pro Epoche: {total_steps}')
print(f'Checkpoint-Intervall: alle {CONFIG["checkpoint_every_steps"]} Steps')

def save_checkpoint(path, state):
    """Checkpoint erst temporär schreiben und danach an die Zielstelle verschieben."""
    temporary_path = path.with_name(f'.{path.name}.tmp')
    torch.save(state, temporary_path)
    os.replace(temporary_path, path)

def make_checkpoint(epoch, next_step, current_epoch_indices, current_running_loss,
                    current_running_count, current_best_val_loss, current_global_step):
    return {
        'checkpoint_version': 2,
        'epoch': epoch,
        'next_step': next_step,
        'epoch_indices': current_epoch_indices,
        'running_loss': current_running_loss,
        'running_count': current_running_count,
        'global_step': current_global_step,
        'best_val_loss': current_best_val_loss,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'model_config': MODEL_CONFIG,
        'dataset_config': DATASET_CONFIG,
        'batch_size': BATCH_SIZE,
        'train_loss': current_running_loss / max(current_running_count, 1),
        'validation_loss': current_best_val_loss,
        'shuffle_generator_state': shuffle_generator.get_state(),
        'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state': torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else None,
        'wandb_run_id': run.id,
    }

for epoch in range(start_epoch, EPOCHS):
    # Reihenfolge der Epoche sichern, damit beim Fortsetzen dieselben Batches folgen.
    if epoch_indices is None:
        epoch_indices = torch.randperm(len(train_dataset), generator=shuffle_generator)
        start_step = 0
        running_loss = 0.0
        running_count = 0

    remaining_indices = epoch_indices[start_step * BATCH_SIZE:].tolist()
    epoch_loader = DataLoader(
        Subset(train_dataset, remaining_indices),
        batch_size=BATCH_SIZE,
        shuffle=False,
        generator=torch.Generator().manual_seed(CONFIG['seed'] + epoch),
    )
    model.train()
    print(f'\nEpoche {epoch + 1}/{EPOCHS} startet bei Step {start_step + 1}/{total_steps}.')

    for offset, (x, y) in enumerate(epoch_loader, start=1):
        step = start_step + offset
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)
        loss = loss_fn(logits.reshape(-1, MODEL_CONFIG['vocab_size']), y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_count += 1
        global_step += 1
        wandb.log({
            'train/loss': loss.item(),
            'epoch': epoch + 1,
            'step_in_epoch': step,
            'global_step': global_step,
        })

        if step % CONFIG['log_every_steps'] == 0 or step == total_steps:
            print(
                f'epoch={epoch + 1}/{EPOCHS}, step={step}/{total_steps}, '
                f'loss={loss.item():.4f}'
            )

        if step % CONFIG['checkpoint_every_steps'] == 0 or step == total_steps:
            checkpoint_state = make_checkpoint(
                epoch, step, epoch_indices, running_loss, running_count,
                best_val_loss, global_step,
            )
            save_checkpoint(latest_checkpoint_path, checkpoint_state)
            print(f'Zwischenstand gespeichert (Epoche {epoch + 1}, Step {step}): {latest_checkpoint_path}')

    train_loss = running_loss / running_count
    print(f'Durchschnittlicher Trainings-Loss: {train_loss:.4f}')

    # Ohne Gradienten validieren; der Validierungs-Loss bestimmt den besten Checkpoint.
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            logits = model(x)
            loss = loss_fn(logits.reshape(-1, MODEL_CONFIG['vocab_size']), y.reshape(-1))
            val_loss += loss.item()
    val_loss /= len(val_loader)
    print(f'Validierungs-Loss: {val_loss:.4f}')
    wandb.log({
        'train/epoch_loss': train_loss,
        'validation/loss': val_loss,
        'epoch': epoch + 1,
        'global_step': global_step,
    })

    improved = val_loss < best_val_loss
    best_val_loss = min(best_val_loss, val_loss)
    # Erst nach der Validierung den Startpunkt für die nächste Epoche sichern.
    checkpoint_state = make_checkpoint(
        epoch + 1, 0, None, 0.0, 0, best_val_loss, global_step,
    )
    if improved:
        save_checkpoint(best_checkpoint_path, checkpoint_state)
        print(f'Neuer bester Checkpoint gespeichert: {best_checkpoint_path}')
    save_checkpoint(latest_checkpoint_path, checkpoint_state)
    print(f'Epochen-Checkpoint gespeichert: {latest_checkpoint_path}')

    epoch_indices = None
    start_step = 0
    running_loss = 0.0
    running_count = 0

run.finish()


## 4. Ein kleiner Text-Check

Damit sehen wir, dass das Modell nach dem Training überhaupt etwas erzeugt.


In [ ]:
model.eval()
prompt = 'The model is'
prompt_ids = tokenizer.encode(prompt)
prompt_tensor = torch.tensor(prompt_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
with torch.no_grad():
    logits = model(prompt_tensor)
    next_token = logits[:, -1, :].argmax(dim=-1).item()
    print('Nächstes Token:', next_token)
    generated = prompt + tokenizer.decode([next_token])
    print('Generiert:', generated)
